# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. We'll extract, inspect, and process the data following the Croissant schema convention.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name')}: {getattr(metadata, 'description')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`. We use Croissant `@id` for every entity.

In [ ]:
# List available record sets and their @id
print("Available record sets (@id):")
for rs in dataset.record_sets:
    print(f"- {rs.id}: {rs.name}")

# Inspect fields for the main record set (typically survey or main table)
# Choose the primary record set (first in list or by known @id)
main_record_set_id = None
if len(dataset.record_sets) > 0:
    main_record_set = dataset.record_sets[0]
    main_record_set_id = main_record_set.id
    print(f"\nFields in record set '{main_record_set_id}':")
    for field in main_record_set.fields:
        print(f"- {field.id}: {field.name} (dataType={field.data_type})")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames, referencing all IDs by their Croissant `@id`.

In [ ]:
# Extract data from all available record sets
record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} records from record set {record_set_id}.")

# Show columns and a sample for the main record set
if main_record_set_id is not None:
    print(f"\nColumns (fields) for {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping by attributes. All field references use `@id` as column names.

In [ ]:
# Example: Filter, normalize, and group on numeric field
# Replace field IDs with actual field @id names from your record set! (See section 2 and 3)

import numpy as np

# Let us try to select a numeric field such as PHQ-9 score or GAD-7 score if present
# For demonstration, let's look for such fields from the list
df = dataframes[main_record_set_id]
field_ids = df.columns.tolist()

# Suppose we have field @id for PHQ-9 sum: 'phq9_sum'
# And group by gender: 'gender'
# Adjust these variables to match your schema's actual field @id

# Try to auto-detect common fields by likely names
field_candidates = [f for f in field_ids if 'phq' in f.lower() and ('sum' in f.lower() or 'score' in f.lower())]
group_candidates = [f for f in field_ids if 'gender' in f.lower() or 'sex' in f.lower()]

if field_candidates:
    numeric_field = field_candidates[0]
else:
    # Default fallback: first numeric-looking field
    possible_numeric = df.select_dtypes(include=np.number).columns.tolist()
    numeric_field = possible_numeric[0] if possible_numeric else field_ids[0]

if group_candidates:
    group_field = group_candidates[0]
else:
    group_field = None

print(f"Using numeric field: {numeric_field}")
if group_field:
    print(f"Grouping by: {group_field}")

# Drop NA
filtered_df = df.dropna(subset=[numeric_field])
filtered_df = filtered_df[filtered_df[numeric_field].apply(lambda x: isinstance(x, (int, float)))]

# Set threshold (e.g. score > 10 for moderate symptoms)
threshold = 10
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by gender (or similar attribute) if available
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nAverage {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between dataset fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the analyzed numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot comparing numeric_field by group_field if available
if group_field and group_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey dataset using Croissant `@id` referencing throughout. We identified core fields and performed initial filtering, normalization, grouping, and visualization. For deeper analysis, consult the dataset's full Croissant schema for precise field interpretations and consider advanced analytical techniques. This workflow can be adapted for AI/ML model preparation or domain-specific research.